# UC-SEARCH-2 — ES Pipeline Health and Geometry Check

**Persona:** Data Engineer

**Goal:** After a catalog reindex, query the Logs API for geometry simplification
events, identify items that fell back to bbox-only representation (`simplification_mode=bbox`),
validate a known item via GeoID lookup, inspect system-level ES bulk failures (sysadmin),
and trace correlated events by `correlation_id`.

**Prerequisites:**
- Catalog has been reindexed (run notebook `01_trigger_full_catalog_reindex` first)
- ES cluster running and reachable
- Admin JWT in `DYNASTORE_ADMIN_TOKEN`; sysadmin JWT in `DYNASTORE_SYSADMIN_TOKEN`
- A known GeoID in `TEST_GEOID` (e.g. `AFR_ETH_ET_AM`)

**Env vars required:** `DYNASTORE_BASE_URL`, `DYNASTORE_ADMIN_TOKEN`,
`DYNASTORE_SYSADMIN_TOKEN`, `CATALOG_ID`, `TEST_GEOID`

**Memory refs:**
- `project_geoid_opensearch_pipeline` — 8 ES indexing bugs fixed 2026-04-02/03; simplify_to_fit
- `project_geoid_logs_es_backend` — DONE f4ff70d; correlation_id middleware; sysadmin auth guards

In [ ]:
import os
import json

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL        = os.environ["DYNASTORE_BASE_URL"]
ADMIN_TOKEN     = os.environ["DYNASTORE_ADMIN_TOKEN"]
SYSADMIN_TOKEN  = os.environ["DYNASTORE_SYSADMIN_TOKEN"]
CATALOG_ID      = os.environ["CATALOG_ID"]
TEST_GEOID      = os.environ.get("TEST_GEOID", "AFR_ETH_ET_AM")

admin_headers = {
    "Authorization": f"Bearer {ADMIN_TOKEN}",
    "Content-Type": "application/json",
}
sysadmin_headers = {
    "Authorization": f"Bearer {SYSADMIN_TOKEN}",
    "Content-Type": "application/json",
}

client   = httpx.Client(base_url=BASE_URL, headers=admin_headers, timeout=30.0)
sysclient = httpx.Client(base_url=BASE_URL, headers=sysadmin_headers, timeout=30.0)

print(f"Connected to {BASE_URL}")
print(f"Catalog: {CATALOG_ID}  Test GeoID: {TEST_GEOID}")

In [ ]:
# Query ES_INDEXING WARNING logs for geometry simplification events
# The obfuscated indexer emits a WARNING when simplify_to_fit triggers,
# documenting which item was simplified and the resulting simplification_mode.

r = client.get(
    f"/logs/catalogs/{CATALOG_ID}",
    params={
        "event_type": "ES_INDEXING",
        "level": "WARNING",
        "limit": 100,
    },
)
print(r.status_code, r.text[:400])
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"

warn_resp = r.json()
warn_entries = warn_resp.get("logs", [])
kibana_url   = warn_resp.get("kibana_dashboard_url")

print(f"\nES_INDEXING WARNING entries: {len(warn_entries)}")
if kibana_url:
    print(f"Kibana dashboard: {kibana_url}")

geom_simplification_entries = [
    e for e in warn_entries
    if "simplif" in str(e.get("message", "")).lower()
]
print(f"Geometry simplification warnings: {len(geom_simplification_entries)}")
for e in geom_simplification_entries[:5]:
    print(f"  {e.get('message', '')[:120]}")

In [ ]:
# Count items that fell back to bbox simplification mode
# When simplify_to_fit cannot reduce the geometry below the ES limit,
# it stores only the bounding box in the ES document (simplification_mode=bbox).
# Spatial queries on these items have reduced precision (bbox-level, not polygon-level).

bbox_items = [
    e for e in warn_entries
    if "bbox" in str(e.get("message", "")).lower()
]
print(f"Items with bbox fallback (simplification_mode=bbox): {len(bbox_items)}")
for e in bbox_items[:5]:
    print(f"  [{e.get('catalog_id')}] {e.get('message', '')[:100]}")
    details = e.get("details") or {}
    if details:
        print(f"    details: {json.dumps(details)[:120]}")

In [ ]:
# Validate the test GeoID is found in the obfuscated ES index
# POST /search/catalogs/{catalog_id}/geoid  with body {"geoids": [...], "limit": N}

r = client.post(
    f"/search/catalogs/{CATALOG_ID}/geoid",
    content=json.dumps({"geoids": [TEST_GEOID], "limit": 10}),
)
print(r.status_code, r.text[:600])
assert r.status_code == 200, (
    f"Expected 200 for GeoID batch lookup, got {r.status_code}: {r.text}"
)

geoid_result = r.json()
matched = geoid_result.get("features", geoid_result.get("items", []))
print(f"\nGeoID '{TEST_GEOID}': {len(matched)} item(s) found in ES index")
assert len(matched) > 0, (
    f"GeoID '{TEST_GEOID}' not found in ES index — reindex may not have completed."
)
for item in matched[:3]:
    print(f"  id={item.get('id')}  collection={item.get('collection')}")

In [ ]:
# Check system-level logs for ES bulk failures (sysadmin only)
# GET /logs/system requires sysadmin scope; admin tokens return 403.

r = sysclient.get(
    "/logs/system",
    params={"level": "ERROR", "limit": 50},
)
print(r.status_code, r.text[:600])
assert r.status_code == 200, (
    f"Expected 200 for system logs (sysadmin), got {r.status_code}: {r.text}"
)

sys_logs = r.json().get("logs", [])
es_bulk_errors = [
    e for e in sys_logs
    if "bulk" in str(e.get("message", "")).lower()
    or "elasticsearch" in str(e.get("message", "")).lower()
    or "es" in str(e.get("event_type", "")).lower()
]
print(f"\nSystem-level ERROR log entries: {len(sys_logs)}")
print(f"ES bulk failure entries       : {len(es_bulk_errors)}")
for e in es_bulk_errors[:3]:
    print(f"  [{e.get('event_type')}] {e.get('message', '')[:120]}")

## Correlation ID tracing

Every request that flows through the GeoID middleware receives a `correlation_id`
injected by `CorrelationIdMiddleware` (commit f4ff70d).  Log entries emitted during
that request carry the same ID in `details.correlation_id`, enabling end-to-end
tracing of a single reindex request across multiple log records.

In [ ]:
# Extract a correlation_id from a warning log entry and filter all logs by it

sample_correlation_id = None
for entry in warn_entries:
    details = entry.get("details") or {}
    cid = details.get("correlation_id")
    if cid:
        sample_correlation_id = cid
        print(f"Found correlation_id: {sample_correlation_id}")
        print(f"  From log entry: [{entry.get('level')}] {entry.get('message', '')[:80]}")
        break

if not sample_correlation_id:
    print("No correlation_id found in warning entries.")
    print("This is normal if the log backend does not populate details.correlation_id.")
    print("Skipping correlation trace — check Kibana for X-Correlation-ID header tracing.")

In [ ]:
# Filter all catalog logs by correlation_id to reconstruct the event chain
# The logs API does not currently expose a correlation_id query param;
# client-side filter on the details field is used here.

if sample_correlation_id:
    r = client.get(
        f"/logs/catalogs/{CATALOG_ID}",
        params={"limit": 100},
    )
    assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"
    all_entries = r.json().get("logs", [])
    correlated = [
        e for e in all_entries
        if (e.get("details") or {}).get("correlation_id") == sample_correlation_id
    ]
    correlated_sorted = sorted(correlated, key=lambda e: e.get("timestamp") or "")
    print(f"Correlated events for {sample_correlation_id}: {len(correlated_sorted)}")
    for e in correlated_sorted:
        print(
            f"  {e.get('timestamp', 'n/a'):30s}  [{e.get('level'):7s}]  "
            f"[{e.get('event_type'):20s}]  {e.get('message', '')[:60]}"
        )
else:
    print("No correlation_id available — skipping correlated event chain.")

## Edge cases

### simplification_mode=bbox — spatial precision caveats

When an item's geometry is too complex to fit within the ES source document size
limit (even after successive Douglas-Peucker simplification passes), the driver
falls back to storing the geometry's bounding box (`bbox`) in the ES document.

Spatial queries (e.g. `intersects`, `within`) on such items use bbox-level
precision, which means:
- False positives: items whose bbox intersects the query polygon but whose actual
  geometry does not.
- False negatives: none (bbox always contains the real geometry).

For high-precision spatial queries, fall back to the OGC Features API
(`/features/catalogs/{id}/collections/{cid}/items`) which queries PostgreSQL
directly with full geometry fidelity.

In [ ]:
print("Bbox fallback spatial precision caveats:")
print(f"  Items with bbox fallback in this reindex: {len(bbox_items)}")
print()
print("  Workaround for high-precision queries:")
print(f"  GET {BASE_URL}/features/catalogs/{CATALOG_ID}/collections/{{cid}}/items")
print("  → PostgreSQL with full geometry fidelity (no simplification)")
print()
print("  To reduce bbox fallback rate: increase ES max_source_size or use geometry pre-simplification")
print("  in the ingestion pipeline before items reach the indexer.")

In [ ]:
# Auth guard: GET /logs/system with admin (non-sysadmin) token → 403

r = client.get(
    "/logs/system",
    params={"level": "ERROR", "limit": 5},
)
print(r.status_code, r.text[:300])
assert r.status_code in (401, 403), (
    f"Expected 401 or 403 for admin token on /logs/system, got {r.status_code}: {r.text}"
)
print(f"{r.status_code} confirmed — /logs/system denied for non-sysadmin token.")

client.close()
sysclient.close()